# Llama-3.1-8B-Instruct Government QA Evaluation — 3-Shot + Concise CoT

**FULL FIXED Kaggle version**

Key fixes:
- Uses `meta-llama/Llama-3.1-8B-Instruct` (the model family this account has access to).
- Reads `HF_TOKEN` securely from Kaggle Secrets and authenticates to Hugging Face.
- Uses 4-bit NF4 quantization for Kaggle GPU memory efficiency.
- Makes the 3 demonstrations match the requested reasoning + final-answer format.
- Constrains reasoning to 2–3 short sentences to prevent runaway generations.
- Adaptive generation budgets: **256 → 512 → 768** tokens.
- Retries rows that are truncated **or** fail to produce the required final-answer marker.
- Saves a partial CSV continuously and resumes automatically after a Kaggle session restart.
- Evaluates only the extracted final answer, not the reasoning text.


In [7]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================

!pip install -q -U "transformers>=4.45" accelerate bitsandbytes \
    sacrebleu rapidfuzz nltk "bert-score==0.3.13"


In [8]:
# ============================================================
# CELL 2 — IMPORTS + PATHS + LOAD TEST AND TRAIN DATA
# ============================================================

import os
import re
import gc
import json
import random
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

TRAIN_JSONL = "/kaggle/input/datasets/akra1234/government/government_chat_train.jsonl"
TEST_CSV = "/kaggle/input/datasets/akra1234/government/merged_test_data.csv"

OUT_DIR = Path("/kaggle/working/Llama/few-shot-cot")
OUT_DIR.mkdir(parents=True, exist_ok=True)
PRED_PATH = OUT_DIR / "prediction_few_shot_cot.csv"
PARTIAL_PATH = OUT_DIR / "prediction_few_shot_cot_partial.csv"
RESULT_PATH = OUT_DIR / "result_few_shot_cot.csv"
SHOTS_PATH = OUT_DIR / "few_shot_examples_cot.csv"

assert os.path.exists(TEST_CSV), f"Test file not found: {TEST_CSV}"
assert os.path.exists(TRAIN_JSONL), f"Training file not found: {TRAIN_JSONL}"

raw_df = pd.read_csv(TEST_CSV).fillna("")

print("Test rows:", len(raw_df))
print("Test columns:", raw_df.columns.tolist())


def choose_column(columns, candidates, required=True):
    lookup = {str(c).lower(): c for c in columns}
    for name in candidates:
        if name.lower() in lookup:
            return lookup[name.lower()]
    if required:
        raise ValueError(
            f"Could not find any of {candidates}. Available columns: {list(columns)}"
        )
    return None


QUESTION_COL = choose_column(
    raw_df.columns,
    ["instruction", "question", "prompt", "query"]
)

GOLD_COL = choose_column(
    raw_df.columns,
    ["output", "gold", "reference", "answer", "target"]
)

INPUT_COL = choose_column(
    raw_df.columns,
    ["input"],
    required=False
)

print("Question column:", QUESTION_COL)
print("Gold column:", GOLD_COL)
print("Optional input column:", INPUT_COL)

df = raw_df.copy()

df["question"] = df[QUESTION_COL].astype(str).str.strip()
df["gold"] = df[GOLD_COL].astype(str).str.strip()

if INPUT_COL is not None and INPUT_COL != QUESTION_COL:
    extra = df[INPUT_COL].astype(str).str.strip()
    df["model_input"] = [
        q if not x else f"{q}\n\nঅতিরিক্ত তথ্য:\n{x}"
        for q, x in zip(df["question"], extra)
    ]
else:
    df["model_input"] = df["question"]

display(df.head(3))


# ------------------------------------------------------------
# Robustly read question-answer pairs from the training JSONL.
# Supports common schemas:
#   {"messages": [{"role":"user",...},{"role":"assistant",...}]}
#   {"instruction": "...", "input": "...", "output": "..."}
#   {"question": "...", "answer": "..."}
#   {"conversations": [{"from":"human",...},{"from":"gpt",...}]}
# ------------------------------------------------------------

def _clean_text(x):
    if x is None:
        return ""
    if isinstance(x, (dict, list)):
        return json.dumps(x, ensure_ascii=False)
    return str(x).strip()


def extract_train_pair(obj):
    # ChatML / OpenAI-style messages
    messages = obj.get("messages")
    if isinstance(messages, list):
        user_text = ""
        assistant_text = ""

        for m in messages:
            if not isinstance(m, dict):
                continue
            role = str(m.get("role", "")).lower().strip()
            content = _clean_text(m.get("content", ""))

            if role == "user" and not user_text:
                user_text = content
            elif role == "assistant" and user_text and not assistant_text:
                assistant_text = content
                break

        if user_text and assistant_text:
            return user_text, assistant_text

    # Alpaca-style / standard instruction-output
    q = ""
    for key in ["instruction", "question", "prompt", "query"]:
        if key in obj and _clean_text(obj.get(key)):
            q = _clean_text(obj.get(key))
            break

    a = ""
    for key in ["output", "answer", "response", "target", "gold", "reference"]:
        if key in obj and _clean_text(obj.get(key)):
            a = _clean_text(obj.get(key))
            break

    extra = _clean_text(obj.get("input", ""))
    if q and extra:
        q = f"{q}\n\nঅতিরিক্ত তথ্য:\n{extra}"

    if q and a:
        return q, a

    # ShareGPT-style conversations
    conversations = obj.get("conversations")
    if isinstance(conversations, list):
        user_text = ""
        assistant_text = ""

        for m in conversations:
            if not isinstance(m, dict):
                continue
            role = str(m.get("from", m.get("role", ""))).lower().strip()
            content = _clean_text(m.get("value", m.get("content", "")))

            if role in {"human", "user"} and not user_text:
                user_text = content
            elif role in {"gpt", "assistant", "bot"} and user_text and not assistant_text:
                assistant_text = content
                break

        if user_text and assistant_text:
            return user_text, assistant_text

    return None


train_records = []

with open(TRAIN_JSONL, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        obj = json.loads(line)
        pair = extract_train_pair(obj)

        if pair is not None:
            q, a = pair
            train_records.append({
                "train_line": line_no,
                "question": q.strip(),
                "answer": a.strip(),
            })

train_df = pd.DataFrame(train_records)

if train_df.empty:
    raise ValueError(
        "No usable question-answer pairs were parsed from the training JSONL. "
        "Inspect the file schema and update extract_train_pair()."
    )

train_df = train_df[
    train_df["question"].astype(str).str.strip().ne("")
    & train_df["answer"].astype(str).str.strip().ne("")
].copy()

print("Parsed train QA pairs:", len(train_df))


# ------------------------------------------------------------
# Remove exact train/test question overlap BEFORE shot selection.
# This avoids accidentally placing the answer to a test question
# directly inside the few-shot prompt.
# ------------------------------------------------------------

BN_TO_EN_OVERLAP = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")


def normalize_question_for_overlap(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.translate(BN_TO_EN_OVERLAP).lower()
    text = re.sub(r"[^\u0980-\u09FFA-Za-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


test_question_norms = set(
    df["model_input"].map(normalize_question_for_overlap)
)

train_df["question_norm"] = train_df["question"].map(
    normalize_question_for_overlap
)

before_overlap_filter = len(train_df)

train_pool_df = train_df[
    ~train_df["question_norm"].isin(test_question_norms)
].copy()

# Avoid duplicate demonstrations with the same normalized question.
train_pool_df = train_pool_df.drop_duplicates(
    subset=["question_norm"],
    keep="first",
).reset_index(drop=True)

removed_overlaps = before_overlap_filter - len(
    train_df[~train_df["question_norm"].isin(test_question_norms)]
)

print("Train/test exact-overlap rows excluded:", removed_overlaps)
print("Unique non-overlapping train pool:", len(train_pool_df))

if len(train_pool_df) < 3:
    raise ValueError(
        "Fewer than 3 non-overlapping training examples remain. "
        "Cannot run a 3-shot experiment safely."
    )


Test rows: 248
Test columns: ['id', 'domain', 'topic', 'question_type', 'instruction', 'input', 'output', 'source_url', 'split', 'source']
Question column: instruction
Gold column: output
Optional input column: input


,id,domain,topic,question_type,instruction,input,output,source_url,split,source,question,gold,model_input
0,nid_003,nid,nid_number_structure,documents,NID আবেদন করতে কী কী ডকুমেন্ট লাগে?,,"প্রিন্টেড আবেদনপত্র, পাসপোর্ট সাইজ ছবি, ১৭ ডিজ...",https://services.nidw.gov.bd/,test,NID,NID আবেদন করতে কী কী ডকুমেন্ট লাগে?,"প্রিন্টেড আবেদনপত্র, পাসপোর্ট সাইজ ছবি, ১৭ ডিজ...",NID আবেদন করতে কী কী ডকুমেন্ট লাগে?
1,nid_006,nid,eligibility,procedure,NID কারা আবেদন করতে পারবে?,,০১ অক্টোবর ২০১০ এর আগে জন্মগ্রহণকারী বাংলাদেশী...,https://services.nidw.gov.bd/,test,NID,NID কারা আবেদন করতে পারবে?,০১ অক্টোবর ২০১০ এর আগে জন্মগ্রহণকারী বাংলাদেশী...,NID কারা আবেদন করতে পারবে?
2,nid_007,nid,new_voter_registration,procedure,যদি আগে ভোটার হয়ে থাকি তাহলে কি আবার আবেদন করত...,,"না, আগে ভোটার হয়ে থাকলে নতুন নিবন্ধনের প্রয়োজন...",https://services.nidw.gov.bd/,test,NID,যদি আগে ভোটার হয়ে থাকি তাহলে কি আবার আবেদন করত...,"না, আগে ভোটার হয়ে থাকলে নতুন নিবন্ধনের প্রয়োজন...",যদি আগে ভোটার হয়ে থাকি তাহলে কি আবার আবেদন করত...


Parsed train QA pairs: 1186
Train/test exact-overlap rows excluded: 250
Unique non-overlapping train pool: 933


In [9]:
# ============================================================
# CELL 3 — AUTHENTICATE + LOAD LLAMA-3.1-8B-INSTRUCT IN 4-BIT
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is required/recommended. In Kaggle: Settings -> Accelerator -> GPU."
    )

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

# Read the Hugging Face token securely from Kaggle Secrets.
# In Kaggle, create a secret named exactly: HF_TOKEN
try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Kaggle Secret 'HF_TOKEN' is not available to this notebook. "
        "Add/enable it under Add-ons -> Secrets and restart the session."
    ) from e

# Authenticate explicitly. The token must have permission to read public gated repos.
login(token=HF_TOKEN, add_to_git_credential=False)
account = whoami(token=HF_TOKEN)
print("Authenticated Hugging Face user:", account.get("name", "unknown"))

# Helpful permission diagnostic for fine-grained tokens.
auth = account.get("auth", {}) if isinstance(account, dict) else {}
access_token_info = auth.get("accessToken", {}) if isinstance(auth, dict) else {}
fine_grained = access_token_info.get("fineGrained", {}) if isinstance(access_token_info, dict) else {}
if fine_grained:
    can_read_gated = fine_grained.get("canReadGatedRepos", None)
    print("Token can read gated repos:", can_read_gated)
    if can_read_gated is False:
        raise RuntimeError(
            "Your Hugging Face fine-grained token cannot read gated repositories. "
            "Edit/create the token and enable read access to public gated repositories you can access."
        )

compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer:", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    low_cpu_mem_usage=True,
)

model.eval()

print("Loaded:", MODEL_NAME)
print("Device:", next(model.parameters()).device)


Authenticated Hugging Face user: arkaarkadutta
Token can read gated repos: True
Loading tokenizer: meta-llama/Llama-3.1-8B-Instruct
Loading model in 4-bit...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: meta-llama/Llama-3.1-8B-Instruct
Device: cuda:0


In [10]:
# ============================================================
# CELL 4 — SELECT 3 FIXED FEW-SHOT DEMONSTRATIONS
# ============================================================

NUM_SHOTS = 3

# Reject only extremely long demonstration pairs so a random outlier
# cannot consume most of the context window. Selection remains seeded.
MAX_DEMO_PAIR_TOKENS = 512

shuffled_train = train_pool_df.sample(
    frac=1.0,
    random_state=SEED,
).reset_index(drop=True)

selected_rows = []

for _, row in shuffled_train.iterrows():
    pair_messages = [
        {"role": "user", "content": str(row["question"])},
        {"role": "assistant", "content": str(row["answer"])},
    ]

    pair_ids = tokenizer.apply_chat_template(
        pair_messages,
        tokenize=True,
        add_generation_prompt=False,
    )

    if len(pair_ids) <= MAX_DEMO_PAIR_TOKENS:
        selected_rows.append(row)

    if len(selected_rows) == NUM_SHOTS:
        break

if len(selected_rows) < NUM_SHOTS:
    raise ValueError(
        f"Could select only {len(selected_rows)} demonstrations under "
        f"MAX_DEMO_PAIR_TOKENS={MAX_DEMO_PAIR_TOKENS}. "
        "Increase the limit or inspect the training data."
    )

shots_df = pd.DataFrame(selected_rows).reset_index(drop=True)
shots_df.insert(0, "shot_id", range(1, NUM_SHOTS + 1))

shots_df[
    ["shot_id", "train_line", "question", "answer"]
].to_csv(
    SHOTS_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(f"Selected {NUM_SHOTS} fixed demonstrations.")
print("Saved demonstration audit:", SHOTS_PATH)

for _, row in shots_df.iterrows():
    print("\n" + "=" * 70)
    print(f"SHOT {int(row['shot_id'])} | train line {int(row['train_line'])}")
    print("- Question:")
    print(row["question"])
    print("- Answer:")
    print(row["answer"])

display(shots_df[["shot_id", "train_line", "question", "answer"]])


Selected 3 fixed demonstrations.
Saved demonstration audit: /kaggle/working/Llama/few-shot-cot/few_shot_examples_cot.csv

SHOT 1 | train line 1056
- Question:
কেন্দ্রীভূত টিআইএন ব্যবস্থার সরকারের জন্য প্রধান সুবিধা কী?
- Answer:
এটি কর আইন কার্যকরভাবে পরিচালনা, কর পরিপালন ভালোভাবে পর্যবেক্ষণ এবং কর্পোরেট আর্থিক কার্যক্রমে অধিক স্বচ্ছতা নিশ্চিত করতে সাহায্য করে।

SHOT 2 | train line 103
- Question:
প্রবাসী ভোটার নির্দেশনা কোথায় পাওয়া যায়?
- Answer:
NID ওয়েবসাইটের ‘প্রবাসী নতুন ভোটার নির্দেশনা’ সেকশনে পাওয়া যায়।

SHOT 3 | train line 818
- Question:
শিশুর জন্মের ৪৫ দিনের মধ্যে জন্ম নিবন্ধন করলে টাকা লাগে?
- Answer:
জন্মের ৪৫ দিনের মধ্যে জন্ম নিবন্ধন করলে ফি লাগে না। এটি বিনামূল্যে করা যায়।


,shot_id,train_line,question,answer
0,1,1056,কেন্দ্রীভূত টিআইএন ব্যবস্থার সরকারের জন্য প্রধ...,"এটি কর আইন কার্যকরভাবে পরিচালনা, কর পরিপালন ভা..."
1,2,103,প্রবাসী ভোটার নির্দেশনা কোথায় পাওয়া যায়?,NID ওয়েবসাইটের ‘প্রবাসী নতুন ভোটার নির্দেশনা’ ...
2,3,818,শিশুর জন্মের ৪৫ দিনের মধ্যে জন্ম নিবন্ধন করলে ...,জন্মের ৪৫ দিনের মধ্যে জন্ম নিবন্ধন করলে ফি লাগ...


In [12]:
# ============================================================
# CELL 5 — 3-SHOT + CONCISE CoT GENERATION
# Consistent demonstrations + final-answer extraction + adaptive retry + resume
# ============================================================

SYSTEM_PROMPT = (
    "আপনি বাংলাদেশের সরকারি সেবা সম্পর্কিত প্রশ্নের সহায়ক। "
    "ব্যবহারকারীর প্রশ্নের উত্তর বাংলায় দিন। "
    "প্রথমে সর্বোচ্চ ২-৩টি সংক্ষিপ্ত বাক্যে প্রাসঙ্গিক তথ্য বিশ্লেষণ করুন। "
    "তারপর অবশ্যই 'চূড়ান্ত উত্তর:' শিরোনাম দিয়ে সরাসরি ও তথ্যভিত্তিক উত্তর দিন। "
    "দীর্ঘ ব্যাখ্যা, পুনরাবৃত্তি, অপ্রয়োজনীয় ভূমিকা বা একই কথা বারবার লিখবেন না। "
    "কোনো reference answer, dataset, training example বা evaluation-এর কথা উল্লেখ করবেন না।"
)

COT_USER_INSTRUCTION = (
    "\n\nএই ফরম্যাটটি ঠিকভাবে অনুসরণ করুন:\n"
    "ধাপে ধাপে বিশ্লেষণ:\n"
    "[সর্বোচ্চ ২-৩টি সংক্ষিপ্ত বাক্য]\n\n"
    "চূড়ান্ত উত্তর:\n"
    "[সংক্ষিপ্ত, সরাসরি উত্তর]"
)

# Generation configuration.
BATCH_SIZE = 4
RETRY_BATCH_SIZE = 1
MAX_INPUT_TOKENS = 4096
GENERATION_BUDGETS = (512,724)

# False is recommended for Kaggle Save & Run All because it resumes from the
# continuously saved partial CSV after an interrupted session.
# Set True manually only when you intentionally want to discard all old outputs.
FORCE_REGENERATE = False

# Continue evaluation if a rare output still reaches the final budget.
FAIL_IF_TRUNCATED = False


def demo_reasoning_text():
    """A neutral formatting cue; it does not inject new factual claims into demonstrations."""
    return (
        "প্রশ্নটির প্রাসঙ্গিক সরকারি সেবার তথ্য সংক্ষেপে বিবেচনা করা হয়েছে। "
        "এখন সরাসরি উত্তর দেওয়া হলো।"
    )


def make_prompt(user_text):
    """
    Build one prompt containing:
      system instruction
      + the same 3 fixed demonstrations
      + the current test question.

    Demonstrations now use the same two-section format requested from the model.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
    ]

    # Same fixed demonstrations for every test item.
    for _, shot in shots_df.iterrows():
        messages.append(
            {"role": "user", "content": str(shot["question"])}
        )
        messages.append(
            {
                "role": "assistant",
                "content": (
                    "ধাপে ধাপে বিশ্লেষণ:\n"
                    + demo_reasoning_text()
                    + "\n\nচূড়ান্ত উত্তর:\n"
                    + str(shot["answer"]).strip()
                ),
            }
        )

    current_user_text = str(user_text).strip() + COT_USER_INSTRUCTION
    messages.append({"role": "user", "content": current_user_text})

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def has_final_answer_marker(text):
    """Check whether the required final-answer boundary is present."""
    text = str(text)
    return bool(
        re.search(r"চূড়ান্ত\s*উত্তর\s*[:：]", text, flags=re.IGNORECASE)
        or re.search(r"final\s+answer\s*[:：]", text, flags=re.IGNORECASE)
    )


def extract_final_answer(text):
    """Extract only the final answer; metrics must not score the reasoning text."""
    text = str(text).strip()
    if not text:
        return ""

    # Bengali marker first.
    matches = list(
        re.finditer(
            r"চূড়ান্ত\s*উত্তর\s*[:：]\s*",
            text,
            flags=re.IGNORECASE,
        )
    )
    if matches:
        final_text = text[matches[-1].end():].strip()
        if final_text:
            return final_text

    # English fallback.
    matches = list(
        re.finditer(
            r"final\s+answer\s*[:：]\s*",
            text,
            flags=re.IGNORECASE,
        )
    )
    if matches:
        final_text = text[matches[-1].end():].strip()
        if final_text:
            return final_text

    # Preserve text for auditing if the marker is missing.
    return text


def extract_cot_reasoning(text):
    """Extract the short reasoning section for audit only; never used by metrics."""
    text = str(text).strip()
    if not text:
        return ""

    final_match = re.search(
        r"চূড়ান্ত\s*উত্তর\s*[:：]",
        text,
        flags=re.IGNORECASE,
    )
    if final_match is None:
        final_match = re.search(
            r"final\s+answer\s*[:：]",
            text,
            flags=re.IGNORECASE,
        )
    if final_match is None:
        return ""

    reasoning = text[:final_match.start()].strip()
    reasoning = re.sub(
        r"^\s*ধাপে\s*ধাপে\s*বিশ্লেষণ\s*[:：]\s*",
        "",
        reasoning,
        flags=re.IGNORECASE,
    )
    reasoning = re.sub(
        r"^\s*(analysis|reasoning)\s*[:：]\s*",
        "",
        reasoning,
        flags=re.IGNORECASE,
    )
    return reasoning.strip()


def _get_eos_token_ids():
    """Collect tokenizer/model EOS and Llama end-of-turn token ids."""
    eos_ids = []

    eos = model.generation_config.eos_token_id
    if eos is None:
        eos = tokenizer.eos_token_id

    if isinstance(eos, int):
        eos_ids.append(int(eos))
    elif eos is not None:
        eos_ids.extend(int(x) for x in eos)

    if tokenizer.eos_token_id is not None:
        eos_ids.append(int(tokenizer.eos_token_id))

    # Llama chat models commonly use <|eot_id|>. Add it when the tokenizer knows it.
    try:
        eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
        if isinstance(eot_id, int) and eot_id >= 0:
            eos_ids.append(int(eot_id))
    except Exception:
        pass

    eos_ids = list(dict.fromkeys(eos_ids))
    if not eos_ids:
        raise ValueError("No EOS/end-of-turn token id is configured.")
    return eos_ids


EOS_TOKEN_IDS = _get_eos_token_ids()
GEN_EOS = EOS_TOKEN_IDS[0] if len(EOS_TOKEN_IDS) == 1 else EOS_TOKEN_IDS
EOS_TOKEN_ID_SET = set(EOS_TOKEN_IDS)

print("Generation EOS token ids:", EOS_TOKEN_IDS)
print("Generation budgets:", GENERATION_BUDGETS)
print(f"Experiment mode: {NUM_SHOTS}-shot + concise CoT")
print("Resume enabled:", not FORCE_REGENERATE)


def generate_batch(texts, max_new_tokens):
    """Generate a batch and return extracted answers plus audit fields."""
    prompts = [make_prompt(x) for x in texts]

    prompt_token_lengths = [
        len(
            tokenizer(
                p,
                add_special_tokens=False,
                truncation=False,
            )["input_ids"]
        )
        for p in prompts
    ]

    longest_prompt = max(prompt_token_lengths)
    if longest_prompt > MAX_INPUT_TOKENS:
        raise RuntimeError(
            "A 3-shot + CoT prompt exceeds MAX_INPUT_TOKENS="
            f"{MAX_INPUT_TOKENS}. Longest prompt has {longest_prompt} tokens. "
            "Increase MAX_INPUT_TOKENS or shorten the demonstrations. "
            "The notebook refuses to silently truncate the current test question."
        )

    batch = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=False,
    )

    input_device = next(model.parameters()).device
    batch = {k: v.to(input_device) for k, v in batch.items()}
    input_width = batch["input_ids"].shape[1]

    with torch.inference_mode():
        generated = model.generate(
            **batch,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=GEN_EOS,
        )

    new_tokens = generated[:, input_width:]
    token_rows = new_tokens.detach().cpu().tolist()

    cot_outputs = tokenizer.batch_decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    cot_outputs = [x.strip() for x in cot_outputs]

    final_answers = [extract_final_answer(x) for x in cot_outputs]
    cot_reasonings = [extract_cot_reasoning(x) for x in cot_outputs]
    final_answer_extracted_flags = [has_final_answer_marker(x) for x in cot_outputs]

    generated_lengths = []
    truncated_flags = []

    for token_ids in token_rows:
        first_eos_position = next(
            (j for j, tok in enumerate(token_ids) if tok in EOS_TOKEN_ID_SET),
            None,
        )
        if first_eos_position is None:
            generated_lengths.append(len(token_ids))
            truncated_flags.append(True)
        else:
            generated_lengths.append(first_eos_position + 1)
            truncated_flags.append(False)

    return (
        final_answers,
        cot_outputs,
        cot_reasonings,
        final_answer_extracted_flags,
        truncated_flags,
        generated_lengths,
    )


def save_partial(frame):
    """Persist progress after every batch/retry so Kaggle runs can resume."""
    frame.sort_values("row_index").to_csv(
        PARTIAL_PATH,
        index=False,
        encoding="utf-8-sig",
    )


# ------------------------------------------------------------
# Clean start / resume
# ------------------------------------------------------------

if FORCE_REGENERATE:
    for old_path in [PARTIAL_PATH, PRED_PATH, RESULT_PATH]:
        if old_path.exists():
            old_path.unlink()
            print("Removed old output:", old_path)

predictions = []

required_cols = {
    "row_index",
    "question",
    "gold",
    "prediction",
    "cot_output",
    "cot_reasoning",
    "final_answer_extracted",
    "truncated",
    "generated_tokens",
    "max_new_tokens_used",
    "generation_attempts",
    "num_shots",
    "shot_train_lines",
    "cot_prompting",
}

if (not FORCE_REGENERATE) and PARTIAL_PATH.exists():
    partial = pd.read_csv(PARTIAL_PATH).fillna("")

    # Resume only a contiguous prefix of rows. This prevents accidental skipping.
    valid_schema = required_cols.issubset(partial.columns)
    valid_length = len(partial) <= len(df)
    valid_indices = False

    if valid_schema and valid_length:
        try:
            indices = partial["row_index"].astype(int).tolist()
            valid_indices = indices == list(range(len(indices)))
        except Exception:
            valid_indices = False

    if valid_schema and valid_length and valid_indices:
        predictions = partial.sort_values("row_index").to_dict("records")
        print(f"Resuming from {len(predictions)} completed rows.")
    else:
        print("Existing partial file is incompatible/non-contiguous; starting fresh.")


# ------------------------------------------------------------
# PASS 1 — initial budget
# ------------------------------------------------------------

initial_budget = GENERATION_BUDGETS[0]
start = len(predictions)

for i in tqdm(
    range(start, len(df), BATCH_SIZE),
    desc=f"Llama {NUM_SHOTS}-shot + CoT generation ({initial_budget} tokens)",
):
    end = min(i + BATCH_SIZE, len(df))
    batch_texts = df.iloc[i:end]["model_input"].tolist()

    (
        answers,
        cot_outputs,
        cot_reasonings,
        final_answer_extracted_flags,
        truncated_flags,
        generated_lengths,
    ) = generate_batch(batch_texts, max_new_tokens=initial_budget)

    for local_idx, (
        answer,
        cot_output,
        cot_reasoning,
        final_answer_extracted,
        truncated,
        gen_len,
    ) in enumerate(
        zip(
            answers,
            cot_outputs,
            cot_reasonings,
            final_answer_extracted_flags,
            truncated_flags,
            generated_lengths,
        )
    ):
        row_idx = i + local_idx
        row = df.iloc[row_idx]

        record = {
            "row_index": row_idx,
            "question": row["question"],
            "gold": row["gold"],
            "prediction": answer,
            "cot_reasoning": cot_reasoning,
            "cot_output": cot_output,
            "final_answer_extracted": bool(final_answer_extracted),
            "truncated": bool(truncated),
            "generated_tokens": int(gen_len),
            "max_new_tokens_used": int(initial_budget),
            "generation_attempts": 1,
            "num_shots": NUM_SHOTS,
            "cot_prompting": True,
            "shot_train_lines": "|".join(
                shots_df["train_line"].astype(int).astype(str).tolist()
            ),
        }

        for col in [
            "id",
            "domain",
            "topic",
            "question_type",
            "source_url",
            "split",
            "source",
        ]:
            if col in df.columns:
                record[col] = row[col]

        predictions.append(record)

    save_partial(pd.DataFrame(predictions))


pred_df = (
    pd.DataFrame(predictions)
    .sort_values("row_index")
    .reset_index(drop=True)
)

assert len(pred_df) == len(df), (
    f"Generated {len(pred_df)} predictions for {len(df)} test rows."
)

print(
    f"After {initial_budget}-token pass, truncated outputs:",
    int(pred_df["truncated"].astype(bool).sum()),
)
print(
    "Outputs missing the requested final-answer marker:",
    int((~pred_df["final_answer_extracted"].astype(bool)).sum()),
)


# ------------------------------------------------------------
# ADAPTIVE RETRY
# Retry rows that are truncated OR missing the required final-answer marker.
# ------------------------------------------------------------

for retry_budget in GENERATION_BUDGETS[1:]:
    needs_retry_mask = (
        pred_df["truncated"].astype(bool)
        | (~pred_df["final_answer_extracted"].astype(bool))
    )
    retry_indices = pred_df.index[needs_retry_mask].tolist()

    if not retry_indices:
        break

    print(
        f"Retrying {len(retry_indices)} problematic outputs "
        f"with max_new_tokens={retry_budget}..."
    )

    for pos in tqdm(
        range(0, len(retry_indices), RETRY_BATCH_SIZE),
        desc=f"CoT retry at {retry_budget} tokens",
    ):
        frame_indices = retry_indices[pos: pos + RETRY_BATCH_SIZE]

        row_indices = (
            pred_df.loc[frame_indices, "row_index"]
            .astype(int)
            .tolist()
        )
        retry_texts = df.iloc[row_indices]["model_input"].tolist()

        (
            answers,
            cot_outputs,
            cot_reasonings,
            final_answer_extracted_flags,
            truncated_flags,
            generated_lengths,
        ) = generate_batch(retry_texts, max_new_tokens=retry_budget)

        for (
            frame_idx,
            answer,
            cot_output,
            cot_reasoning,
            final_answer_extracted,
            truncated,
            gen_len,
        ) in zip(
            frame_indices,
            answers,
            cot_outputs,
            cot_reasonings,
            final_answer_extracted_flags,
            truncated_flags,
            generated_lengths,
        ):
            pred_df.at[frame_idx, "prediction"] = answer
            pred_df.at[frame_idx, "cot_output"] = cot_output
            pred_df.at[frame_idx, "cot_reasoning"] = cot_reasoning
            pred_df.at[frame_idx, "final_answer_extracted"] = bool(final_answer_extracted)
            pred_df.at[frame_idx, "truncated"] = bool(truncated)
            pred_df.at[frame_idx, "generated_tokens"] = int(gen_len)
            pred_df.at[frame_idx, "max_new_tokens_used"] = int(retry_budget)
            pred_df.at[frame_idx, "generation_attempts"] = (
                int(pred_df.at[frame_idx, "generation_attempts"]) + 1
            )

        save_partial(pred_df)

    print(
        f"After {retry_budget}-token retry — truncated:",
        int(pred_df["truncated"].astype(bool).sum()),
        "| missing final marker:",
        int((~pred_df["final_answer_extracted"].astype(bool)).sum()),
    )


# ------------------------------------------------------------
# Final validation + save predictions
# ------------------------------------------------------------

remaining_truncated = int(pred_df["truncated"].astype(bool).sum())
missing_final_marker = int((~pred_df["final_answer_extracted"].astype(bool)).sum())

if remaining_truncated:
    truncated_debug = pred_df.loc[
        pred_df["truncated"].astype(bool),
        [
            "row_index",
            "question",
            "prediction",
            "cot_output",
            "generated_tokens",
            "max_new_tokens_used",
            "generation_attempts",
        ],
    ]

    debug_path = OUT_DIR / "still_truncated_after_retry_few_shot_cot.csv"
    truncated_debug.to_csv(debug_path, index=False, encoding="utf-8-sig")
    print("Saved remaining-truncation audit:", debug_path)

    if FAIL_IF_TRUNCATED:
        raise RuntimeError(
            f"{remaining_truncated} outputs are still truncated after "
            f"max_new_tokens={GENERATION_BUDGETS[-1]}."
        )
    else:
        print(
            f"WARNING: {remaining_truncated} prediction(s) still reached the "
            f"{GENERATION_BUDGETS[-1]}-token limit and remain flagged."
        )

if missing_final_marker:
    marker_debug_path = OUT_DIR / "missing_final_answer_marker_few_shot_cot.csv"
    pred_df.loc[
        ~pred_df["final_answer_extracted"].astype(bool),
        [
            "row_index",
            "question",
            "gold",
            "prediction",
            "cot_output",
            "generated_tokens",
            "max_new_tokens_used",
            "generation_attempts",
        ],
    ].to_csv(marker_debug_path, index=False, encoding="utf-8-sig")
    print("Saved final-answer-marker audit:", marker_debug_path)

# Final prediction file used by downstream evaluation cells.
pred_df.to_csv(PRED_PATH, index=False, encoding="utf-8-sig")
# Keep partial synchronized with the final state as well.
save_partial(pred_df)

print("\nGeneration complete:", len(pred_df))
print("Truncated outputs:", remaining_truncated)
print("Missing final-answer marker:", missing_final_marker)
print("Saved predictions:", PRED_PATH)
print("Saved resumable state:", PARTIAL_PATH)
print(
    "Maximum generation budget actually used:",
    int(pred_df["max_new_tokens_used"].max()),
)

display(
    pred_df[
        [
            "question",
            "gold",
            "cot_reasoning",
            "prediction",
            "final_answer_extracted",
            "truncated",
            "generated_tokens",
            "max_new_tokens_used",
            "generation_attempts",
        ]
    ].head(3)
)


Generation EOS token ids: [128001, 128008, 128009]
Generation budgets: (512, 724)
Experiment mode: 3-shot + concise CoT
Resume enabled: True
Resuming from 40 completed rows.


Llama 3-shot + CoT generation (512 tokens):   0%|          | 0/52 [00:00<?, ?it/s]

After 512-token pass, truncated outputs: 36
Outputs missing the requested final-answer marker: 31
Retrying 55 problematic outputs with max_new_tokens=724...


CoT retry at 724 tokens:   0%|          | 0/55 [00:00<?, ?it/s]

After 724-token retry — truncated: 13 | missing final marker: 24
Saved remaining-truncation audit: /kaggle/working/Llama/few-shot-cot/still_truncated_after_retry_few_shot_cot.csv
Saved final-answer-marker audit: /kaggle/working/Llama/few-shot-cot/missing_final_answer_marker_few_shot_cot.csv

Generation complete: 248
Truncated outputs: 13
Missing final-answer marker: 24
Saved predictions: /kaggle/working/Llama/few-shot-cot/prediction_few_shot_cot.csv
Saved resumable state: /kaggle/working/Llama/few-shot-cot/prediction_few_shot_cot_partial.csv
Maximum generation budget actually used: 724


,question,gold,cot_reasoning,prediction,final_answer_extracted,truncated,generated_tokens,max_new_tokens_used,generation_attempts
0,NID আবেদন করতে কী কী ডকুমেন্ট লাগে?,"প্রিন্টেড আবেদনপত্র, পাসপোর্ট সাইজ ছবি, ১৭ ডিজ...",প্রশ্নটির প্রাসঙ্গিক সরকারি সেবার তথ্য সংক্ষেপ...,প্রাপ্ত বয়সী নাগরিককে নাগরিক আইডেন্টিটি কার্ড...,True,True,724,724,2
1,NID কারা আবেদন করতে পারবে?,০১ অক্টোবর ২০১০ এর আগে জন্মগ্রহণকারী বাংলাদেশী...,প্রশ্নটির প্রাসঙ্গিক সরকারি সেবার তথ্য সংক্ষেপ...,বাংলাদেশের নাগরিক আবেদন করতে পারবে।,True,False,202,256,1
2,যদি আগে ভোটার হয়ে থাকি তাহলে কি আবার আবেদন করত...,"না, আগে ভোটার হয়ে থাকলে নতুন নিবন্ধনের প্রয়োজন...",প্রশ্নটির প্রাসঙ্গিক সরকারি সেবার তথ্য সংক্ষেপ...,যদি আগে ভোটার হয়ে থাকি তাহলে আবার আবেদন করতে হ...,True,False,224,256,1


In [ ]:
# ============================================================
# CELL 6 — FREE LLAMA GPU MEMORY BEFORE BERTSCORE
# ============================================================

del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Llama model removed from memory.")


In [14]:
# ============================================================
# CELL 7 — METRIC FUNCTIONS
# Keeps the same normalization / Token F1 / ROUGE definitions
# as the previous evaluation pipeline.
# ============================================================

BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)


def normalize(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    text = text.translate(
        BN_TO_EN
    ).lower()

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def tokens(text):
    return normalize(text).split()


# ---------------- Normalized Exact Match ----------------

def normalized_exact_match(pred, gold):

    return float(
        normalize(pred)
        ==
        normalize(gold)
    )


# ---------------- Token F1 ----------------

def token_f1(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p and not g:
        return 1.0

    if not p or not g:
        return 0.0

    overlap = sum(
        (
            Counter(p)
            &
            Counter(g)
        ).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ---------------- ROUGE-N F1 ----------------

def rouge_n(pred, gold, n):

    p = tokens(pred)
    g = tokens(gold)

    if len(p) < n or len(g) < n:
        return 0.0

    pg = Counter(
        tuple(p[i:i+n])
        for i in range(len(p)-n+1)
    )

    gg = Counter(
        tuple(g[i:i+n])
        for i in range(len(g)-n+1)
    )

    overlap = sum(
        (pg & gg).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / sum(pg.values())
    recall = overlap / sum(gg.values())

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ---------------- ROUGE-L F1 ----------------

def rouge_l(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p and not g:
        return 1.0

    if not p or not g:
        return 0.0

    dp = [0] * (len(g) + 1)

    for x in p:

        new = [0]

        for j, y in enumerate(g, 1):

            if x == y:
                new.append(dp[j-1] + 1)

            else:
                new.append(
                    max(dp[j], new[-1])
                )

        dp = new

    lcs = dp[-1]

    precision = lcs / len(p)
    recall = lcs / len(g)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ---------------- Bengali-safe METEOR ----------------
# NLTK's default METEOR uses English Porter stemming + English WordNet.
# For Bangla evaluation, disable those English-only lexical resources while
# retaining METEOR's exact-token alignment and fragmentation penalty.

class IdentityStemmer:
    def stem(self, word):
        return word


class EmptyWordNet:
    def synsets(self, word):
        return []


IDENTITY_STEMMER = IdentityStemmer()
EMPTY_WORDNET = EmptyWordNet()


def meteor_bn(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p and not g:
        return 1.0

    if not p or not g:
        return 0.0

    return meteor_score(
        [g],
        p,
        stemmer=IDENTITY_STEMMER,
        wordnet=EMPTY_WORDNET,
    )


In [15]:
# ============================================================
# CELL 8 — COMPUTE ROW-LEVEL METRICS
# ============================================================

eval_df = pd.read_csv(PRED_PATH).fillna("")

# Audit any generation that reached the final 1024-token budget.
# We continue evaluation because the rare truncated case is intentionally retained
# and its count is reported separately in CELL 10.
if "truncated" in eval_df.columns:
    n_truncated_for_eval = int(
        eval_df["truncated"].astype(str).str.lower().eq("true").sum()
    )
else:
    n_truncated_for_eval = 0

print(f"Evaluating all {len(eval_df)} predictions...")
print(f"Predictions marked truncated: {n_truncated_for_eval}")

if n_truncated_for_eval:
    print(
        "NOTE: These rows are included in the aggregate metrics and the "
        "truncated-output count is reported separately."
    )

eval_df["Normalized Exact Match"] = [
    normalized_exact_match(p, g)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["Token F1"] = [
    token_f1(p, g)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["Fuzzy Match"] = [
    fuzz.token_set_ratio(
        normalize(p),
        normalize(g)
    ) / 100
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["ROUGE-1"] = [
    rouge_n(p, g, 1)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["ROUGE-2"] = [
    rouge_n(p, g, 2)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["ROUGE-L"] = [
    rouge_l(p, g)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["METEOR"] = [
    meteor_bn(p, g)
    for p, g in tqdm(
        zip(
            eval_df["prediction"],
            eval_df["gold"]
        ),
        total=len(eval_df),
        desc="METEOR"
    )
]


# ============================================================
# CORPUS BLEU
# ============================================================

bleu = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True
)

pred_texts = [
    " ".join(tokens(x))
    for x in eval_df["prediction"]
]

gold_texts = [
    " ".join(tokens(x))
    for x in eval_df["gold"]
]

corpus_bleu = (
    bleu.corpus_score(
        pred_texts,
        [gold_texts]
    ).score
    / 100
)

print("Corpus BLEU:", corpus_bleu)


Evaluating all 248 predictions...
Predictions marked truncated: 13
NOTE: These rows are included in the aggregate metrics and the truncated-output count is reported separately.


METEOR:   0%|          | 0/248 [00:00<?, ?it/s]

Corpus BLEU: 0.03625541688838032


In [16]:
# ============================================================
# CELL 9 — BERTSCORE
# model: bert-base-multilingual-cased
# ============================================================

print("Calculating multilingual BERTScore...")

bert_device = "cuda" if torch.cuda.is_available() else "cpu"
bert_batch_size = 8 if torch.cuda.is_available() else 4

P, R, F1 = bert_score(
    eval_df["prediction"].astype(str).tolist(),
    eval_df["gold"].astype(str).tolist(),
    model_type="bert-base-multilingual-cased",
    batch_size=bert_batch_size,
    device=bert_device,
    idf=False,
    rescale_with_baseline=False,
    verbose=True
)

eval_df["BERTScore Precision"] = P.cpu().numpy()
eval_df["BERTScore Recall"] = R.cpu().numpy()
eval_df["BERTScore F1"] = F1.cpu().numpy()

print("BERTScore complete.")


Calculating multilingual BERTScore...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/51 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/31 [00:00<?, ?it/s]

done in 4.61 seconds, 53.80 sentences/sec
BERTScore complete.


In [17]:
# ============================================================
# CELL 10 — FINAL RESULT + SAVE FEW-SHOT OUTPUTS
# ============================================================

truncated_count = int(
    eval_df["truncated"]
    .astype(str)
    .str.lower()
    .eq("true")
    .sum()
)

empty_output_count = int(
    eval_df["prediction"].astype(str).str.strip().eq("").sum()
)

avg_generated_tokens = (
    pd.to_numeric(eval_df["generated_tokens"], errors="coerce").mean()
    if "generated_tokens" in eval_df.columns else np.nan
)

max_generated_tokens = (
    pd.to_numeric(eval_df["generated_tokens"], errors="coerce").max()
    if "generated_tokens" in eval_df.columns else np.nan
)

final_answer_marker_failures = (
    int((~eval_df["final_answer_extracted"].astype(bool)).sum())
    if "final_answer_extracted" in eval_df.columns
    else np.nan
)


result = pd.DataFrame({
    "metric": [
        "Normalized Exact Match",
        "Token F1",
        "Fuzzy Match",
        "Corpus BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "METEOR",
        "BERTScore Precision",
        "BERTScore Recall",
        "BERTScore F1",
        "Truncated Outputs",
        "Empty Outputs",
        "Average Generated Tokens",
        "Maximum Generated Tokens",
        "Final-Answer Marker Failures",
    ],
    "score": [
        eval_df["Normalized Exact Match"].mean(),
        eval_df["Token F1"].mean(),
        eval_df["Fuzzy Match"].mean(),
        corpus_bleu,
        eval_df["ROUGE-1"].mean(),
        eval_df["ROUGE-2"].mean(),
        eval_df["ROUGE-L"].mean(),
        eval_df["METEOR"].mean(),
        eval_df["BERTScore Precision"].mean(),
        eval_df["BERTScore Recall"].mean(),
        eval_df["BERTScore F1"].mean(),
        truncated_count,
        empty_output_count,
        avg_generated_tokens,
        max_generated_tokens,
        final_answer_marker_failures,
    ]
})

# Save row-level predictions + all row-level metrics.
eval_df.to_csv(
    PRED_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Save aggregate metrics.
result.to_csv(
    RESULT_PATH,
    index=False,
    encoding="utf-8-sig"
)

display(result)

print(f"\nExperiment: {NUM_SHOTS}-shot + explicit CoT, seed={SEED}")
print("Selected shot train lines:", shots_df["train_line"].astype(int).tolist())
print("\nSaved:")
print(PRED_PATH)
print(RESULT_PATH)

print("\nValidation:")
print("Rows evaluated:", len(eval_df))
print("Truncated outputs:", truncated_count)
print("Empty outputs:", empty_output_count)
print("Final-answer marker failures:", final_answer_marker_failures)

print("\nNOTE:")
print("/kaggle/input is read-only. Kaggle outputs must be written under /kaggle/working.")


,metric,score
0,Normalized Exact Match,0.000000
1,Token F1,0.217106
2,Fuzzy Match,0.527417
3,Corpus BLEU,0.036255
4,ROUGE-1,0.217106
5,ROUGE-2,0.082645
6,ROUGE-L,0.198549
7,METEOR,0.156470
8,BERTScore Precision,0.740716
9,BERTScore Recall,0.707006



Experiment: 3-shot + explicit CoT, seed=42
Selected shot train lines: [1056, 103, 818]

Saved:
/kaggle/working/Llama/few-shot-cot/prediction_few_shot_cot.csv
/kaggle/working/Llama/few-shot-cot/result_few_shot_cot.csv

Validation:
Rows evaluated: 248
Truncated outputs: 13
Empty outputs: 0
Final-answer marker failures: 24

NOTE:
/kaggle/input is read-only. Kaggle outputs must be written under /kaggle/working.
